# Phase 3 — DPO Results Only (Load Pre-trained Adapters)

**Purpose:** Skip training — load all 5 DPO trial adapters from `MyDrive/dpo_trials` and produce a complete `dpo_results.csv` with BLEU and BERTScore.

**Prerequisite:** All 5 trial folders (D1–D5) must exist in `MyDrive/dpo_trials/`.

**Runtime:** T4 GPU

In [ ]:
# Cell 1: Install dependencies
!pip install -q --no-cache-dir \
    numpy==1.26.4 \
    pandas==2.2.2 \
    transformers==4.44.2 \
    trl==0.9.6 \
    peft==0.12.0 \
    accelerate==0.34.2 \
    bitsandbytes==0.46.0 \
    triton==3.6.0 \
    datasets==2.21.0 \
    evaluate==0.4.3 \
    bert-score==0.3.13 \
    sacrebleu==2.4.3

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3: Imports
import gc
import glob
import json
import os
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import evaluate
import pandas as pd
import torch
from bert_score import score as bert_score_fn
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 4: Config
MODEL_ID          = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
BEST_SFT_ADAPTER_PATH = "/content/drive/MyDrive/sft_trials/T5"  # update to your best SFT trial
PROMPTS_PATH      = "/content/drive/MyDrive/Colab Notebooks/data/test_prompts.json"
DRIVE_TRIALS_DIR  = "/content/drive/MyDrive/dpo_trials"   # root folder containing D1..D5
DPO_RESULTS_PATH  = "/content/drive/MyDrive/dpo_results.csv"
BASELINE_CSV      = "/content/drive/MyDrive/baseline_results.csv"
SFT_RESULTS_CSV   = "/content/drive/MyDrive/sft_results.csv"
MAX_NEW_TOKENS    = 200
TEMPERATURE       = 0.7
TOP_P             = 0.9

# DPO trial metadata for CSV record-keeping
DPO_TRIAL_CONFIGS = [
    {"name": "D1", "desc": "beta=0.1, lr=1e-5, bs=2, 1 epoch",       "beta": 0.1, "lr": 1e-5,  "batch_size": 2, "grad_accum": 4, "epochs": 1},
    {"name": "D2", "desc": "beta=0.1, lr=5e-5, bs=2, 2 epochs",      "beta": 0.1, "lr": 5e-5,  "batch_size": 2, "grad_accum": 4, "epochs": 2},
    {"name": "D3", "desc": "beta=0.3, lr=1e-5, bs=2, 1 epoch",       "beta": 0.3, "lr": 1e-5,  "batch_size": 2, "grad_accum": 4, "epochs": 1},
    {"name": "D4", "desc": "beta=0.5, lr=2e-5, bs=2, 2 epochs",      "beta": 0.5, "lr": 2e-5,  "batch_size": 2, "grad_accum": 4, "epochs": 2},
    {"name": "D5", "desc": "beta=0.1, lr=1e-4, bs=2, 1 epoch (high LR)", "beta": 0.1, "lr": 1e-4, "batch_size": 2, "grad_accum": 4, "epochs": 1},
]

print(f"Trials to evaluate: {[c['name'] for c in DPO_TRIAL_CONFIGS]}")
print(f"Adapter root: {DRIVE_TRIALS_DIR}")

In [ ]:
# Cell 5: Load test prompts + sacrebleu
with open(PROMPTS_PATH) as f:
    data = json.load(f)

test_prompts    = [p["prompt"]    for p in data["prompts"]]
test_references = [p["reference"] for p in data["prompts"]]

empty = [i + 1 for i, r in enumerate(test_references) if not r.strip()]
if empty:
    raise ValueError(f"Missing references for prompt IDs: {empty}")

sacrebleu = evaluate.load("sacrebleu")
print(f"Loaded {len(test_prompts)} test prompts.")

In [ ]:
# Cell 6: Helpers
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)


def evaluate_model(model, tokenizer, prompts, references):
    """Run BLEU + BERTScore on test prompts."""
    model.eval()
    tokenizer.padding_side = "left"
    generated = []
    for prompt in prompts:
        formatted = (
            "Below is an instruction that describes a task. Write a response that "
            "appropriately completes the request.\n\n"
            f"### Instruction:\n{prompt}\n\n### Response:\n"
        )
        inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[1]
        with torch.inference_mode():
            out = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                pad_token_id=tokenizer.eos_token_id,
            )
        text = tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()
        generated.append(text)

    bleu_scores = [
        sacrebleu.compute(predictions=[g], references=[[r]])["score"]
        for g, r in zip(generated, references)
    ]
    _, _, F1 = bert_score_fn(
        generated, references, lang="en", rescale_with_baseline=True, verbose=False
    )
    bert_f1 = [f.item() for f in F1]

    return (
        round(sum(bleu_scores) / len(bleu_scores), 4),
        round(sum(bert_f1) / len(bert_f1), 4),
        generated,
    )


def get_val_loss(trial_dir):
    """Read last eval_loss from trainer_state.json inside checkpoint subfolder."""
    pattern = os.path.join(trial_dir, "checkpoint-*", "trainer_state.json")
    matches = sorted(
        glob.glob(pattern),
        key=lambda p: int(os.path.basename(os.path.dirname(p)).split("-")[1]),
    )
    if not matches:
        return None
    with open(matches[-1]) as f:
        state = json.load(f)
    eval_entries = [e for e in state.get("log_history", []) if "eval_loss" in e]
    return round(eval_entries[-1]["eval_loss"], 4) if eval_entries else None


print("Helpers defined.")

In [ ]:
# Cell 7: Evaluation loop
# Loads each DPO adapter on top of the SFT base, evaluates, then frees GPU memory.
results = []

for cfg in DPO_TRIAL_CONFIGS:
    trial_name   = cfg["name"]
    adapter_path = os.path.join(DRIVE_TRIALS_DIR, trial_name)

    print(f"\n{'='*70}")
    print(f"  DPO TRIAL {trial_name}: {cfg['desc']}")
    print(f"  Adapter: {adapter_path}")
    print(f"{'='*70}")

    if not os.path.isdir(adapter_path):
        print(f"  WARNING: {adapter_path} not found — skipping.")
        continue

    # Tokenizer always from base model (Drive copy may be corrupted from cp)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    # Load SFT base + DPO adapter
    # DPO adapters were trained on top of the SFT adapter, so we load:
    #   base model -> SFT adapter -> DPO adapter (two nested PeftModels)
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, device_map="auto",
    )
    sft_model = PeftModel.from_pretrained(base_model, BEST_SFT_ADAPTER_PATH)
    model     = PeftModel.from_pretrained(sft_model, adapter_path)
    model.eval()

    print("Evaluating on test prompts...")
    mean_bleu, mean_bert_f1, generated_texts = evaluate_model(
        model, tokenizer, test_prompts, test_references
    )

    val_loss = get_val_loss(adapter_path)

    results.append({
        "trial":           trial_name,
        "desc":            cfg["desc"],
        "beta":            cfg["beta"],
        "lr":              cfg["lr"],
        "batch_size":      cfg["batch_size"],
        "grad_accum":      cfg["grad_accum"],
        "epochs":          cfg["epochs"],
        "val_loss":        val_loss,
        "mean_bleu":       mean_bleu,
        "mean_bert_f1":    mean_bert_f1,
        "adapter_path":    adapter_path,
        "generated_texts": generated_texts,
    })

    print(f"\n  {trial_name} | val_loss={val_loss} | BLEU={mean_bleu} | BERTScore F1={mean_bert_f1}")

    del model, sft_model, base_model
    gc.collect()
    torch.cuda.empty_cache()

print(f"\nAll {len(results)} trials evaluated.")

In [ ]:
# Cell 8: Results table + save dpo_results.csv
cols = ["trial", "desc", "beta", "lr", "batch_size", "grad_accum",
        "epochs", "val_loss", "mean_bleu", "mean_bert_f1"]

df_dpo = pd.DataFrame(results)[cols]
pd.set_option("display.max_colwidth", 50)
print(df_dpo.to_string(index=False))

df_dpo.to_csv(DPO_RESULTS_PATH, index=False)
print(f"\nSaved to {DPO_RESULTS_PATH}")

In [ ]:
# Cell 9: Select best DPO trial
df_ranked = df_dpo.sort_values(
    by=["mean_bert_f1", "mean_bleu", "val_loss"],
    ascending=[False, False, True],
    na_position="last",
).reset_index(drop=True)

best     = df_ranked.iloc[0]
best_result = next(r for r in results if r["trial"] == best["trial"])

print("\n" + "="*60)
print(f"  BEST DPO TRIAL: {best['trial']}")
print(f"  Description:    {best['desc']}")
print(f"  BERTScore F1:   {best['mean_bert_f1']}")
print(f"  BLEU:           {best['mean_bleu']}")
print(f"  Val Loss:       {best['val_loss']}")
print(f"  Adapter path:   {best_result['adapter_path']}")
print("="*60)

In [ ]:
# Cell 10: Three-way comparison — Base vs Best SFT vs Best DPO
try:
    df_baseline = pd.read_csv(BASELINE_CSV)
    df_sft      = pd.read_csv(SFT_RESULTS_CSV)

    best_sft = df_sft.sort_values(
        by=["mean_bert_f1", "mean_bleu", "val_loss"], ascending=[False, False, True]
    ).iloc[0]

    print("\nTHREE-WAY METRIC COMPARISON")
    print("-" * 52)
    print(f"Base model      | BLEU: {df_baseline['bleu'].mean():.4f} | BERTScore F1: {df_baseline['bert_f1'].mean():.4f}")
    print(f"Best SFT ({best_sft['trial']})   | BLEU: {best_sft['mean_bleu']:.4f} | BERTScore F1: {best_sft['mean_bert_f1']:.4f}")
    print(f"Best DPO ({best['trial']})   | BLEU: {best['mean_bleu']:.4f} | BERTScore F1: {best['mean_bert_f1']:.4f}")

except FileNotFoundError as e:
    print(f"Could not load comparison CSVs: {e}")
    print(f"Best DPO — BLEU: {best['mean_bleu']} | BERTScore F1: {best['mean_bert_f1']}")

In [ ]:
# Cell 11: Print generated outputs from best DPO trial (for report)
print(f"Generated outputs from best DPO trial ({best['trial']}):")
for i, (prompt, gen, ref) in enumerate(
    zip(test_prompts, best_result["generated_texts"], test_references)
):
    print(f"\n[{i+1}] {prompt[:80]}")
    print(f"  DPO Output: {gen[:200]}")
    print(f"  Reference:  {ref[:200]}")